In [20]:
pip install pandas numpy matplotlib seaborn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [21]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

In [22]:
import ee
import geemap.foliumap as geemap

In [23]:
import ee

ee.Authenticate()
ee.Initialize(project='project-3b20b423-4303-445c-84c')

In [24]:
import geemap.foliumap as geemap

m = geemap.Map()
m

In [25]:
import ee

In [26]:

ee.Authenticate()

True

In [27]:
ee.Initialize(project='project-3b20b423-4303-445c-84c')

In [28]:
import ee

delhi = ee.Geometry.Rectangle([76.84, 28.40, 77.34, 28.88])
modis = ee.ImageCollection("MODIS/061/MOD11A2") \
    .select("LST_Day_1km")
recent = modis.filterDate("2025-01-01", "2026-01-01").mean()
proj = ee.Projection('EPSG:3857')

grid = delhi.coveringGrid(proj, 10000)  # 1 km grid
grid_fc = ee.FeatureCollection(grid)
grid_styled = grid_fc.style(
    color='black',
    width=1,
    fillColor='00000000'
)

Map.addLayer(grid_styled, {}, "Grid")

NameError: name 'Map' is not defined

In [ ]:
lst_c = recent.multiply(0.02).subtract(273.15)
delhi_temp = lst_c.clip(delhi)
stats = delhi_temp.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=delhi,
    scale=1000,
    maxPixels=1e9
)

print(stats.getInfo())

{'LST_Day_1km': 26.567842395481385}


In [ ]:
modis = ee.ImageCollection("MODIS/061/MOD11A2") \
    .select("LST_Day_1km")

lst = modis.filterDate("2025-01-01", "2026-01-01").mean()

lst_c = lst.multiply(0.02).subtract(273.15).clip(delhi)

In [ ]:
def add_temp(feature):
    mean_dict = lst_c.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry(),
        scale=1000,
        maxPixels=1e9
    )
    
    return feature.set({
        'temp': mean_dict.get('LST_Day_1km')
    })

grid_with_temp = grid.map(add_temp)

In [ ]:
top5 = grid_with_temp.sort('temp', False).limit(5)

print(top5.getInfo())

{'type': 'FeatureCollection', 'columns': {'temp': 'Object'}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'EPSG:3857'}}, 'type': 'Polygon', 'coordinates': [[[8550000, 3290000], [8560000, 3290000], [8560000, 3300000], [8550000, 3300000], [8550000, 3290000]]]}, 'id': '855,329', 'properties': {'temp': 28.073306425923324}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'EPSG:3857'}}, 'type': 'Polygon', 'coordinates': [[[8560000, 3290000], [8570000, 3290000], [8570000, 3300000], [8560000, 3300000], [8560000, 3290000]]]}, 'id': '856,329', 'properties': {'temp': 27.707333333333366}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'EPSG:3857'}}, 'type': 'Polygon', 'coordinates': [[[8550000, 3300000], [8560000, 3300000], [8560000, 3310000], [8550000, 3310000], [8550000, 3300000]]]}, 'id': '855,330', 'properties': {'temp': 27.

In [ ]:
import geemap

Map = geemap.Map()
Map = geemap.Map()
Map.centerObject(delhi, 10)

# 1. Heatmap
Map.addLayer(lst_c, vis, "Heatmap")

# 2. Grid (optional)
Map.addLayer(grid_styled, {}, "Grid")



In [ ]:
vis = {
    'min': 27,
    'max': 29,
    'palette': ['blue', 'cyan', 'yellow', 'orange', 'red'],
    'opacity':0.6
}
hotspots = top5.style(
    color='red',
    fillColor='FF000040',  # light transparent red
    width=2
)
# 3. Hotspots
Map.addLayer(hotspots, {}, "Hotspots")


Map


Map(center=[28.640046420006694, 77.09000000000006], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
Map


Map(center=[28.640046420006694, 77.09000000000006], controls=(WidgetControl(options=['position', 'transparent_…